# Graph Feature Engineering

Converts SMILES strings to `torch_geometric.data.Data` molecular graphs and saves them
to `data/processed/graphs.pt`. Node features include Gasteiger partial charges;
edge features include bond type, conjugation, ring membership, and electronegativity
difference. A virtual node is appended to each graph for global information propagation.

In [1]:
import polars as pl
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, rdPartialCharges
from torch_geometric.data import Data

In [2]:
df = pl.read_parquet("../data/processed/baseline_model_features.parquet")
print(df.shape)
df.head()

(3024, 32)


key,name,smiles,mpC,formula,C_count,H_count,O_count,N_count,S_count,F_count,Cl_count,Br_count,I_count,molecular_weight,branch_count,cycle_count,aromatic_ring_count,double_bond_count,triple_bond_count,carboxylic_acid_count,alcohol_count,carbonyl_count,prim_sec_amine_count,amide_count,hbd_count,hba_count,logp,tpsa,rotatable_bond_count,frac_csp3,split
i64,str,str,f64,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,i32,f64,str
27956,"""cyclobutylmethane""","""C1(CCC1)C""",-161.51,"""C5H10""",5,10,0,0,0,0,0,0,0,70.135,0,1,0,0,0,0,0,0,0,0,0,0,1.8064,0.0,0,1.0,"""train"""
16005,"""Nitrogen oxide""","""[O-][N+]#N""",-90.8,"""N2O""",0,0,1,2,0,0,0,0,0,44.013,0,0,0,0,1,0,0,0,0,0,0,2,0.33728,51.21,0,0.0,"""train"""
16127,"""Sulfuryl difluoride""","""FS(F)(=O)=O""",-135.8,"""F2O2S""",0,0,2,0,1,2,0,0,0,102.061,1,0,0,2,0,0,0,0,0,0,0,2,0.1702,34.14,0,0.0,"""train"""
17138,"""disopyramide""","""CC(C)N(CCC(c1ccccn1)(c2ccccc2)…",94.8,"""C21H29N3O""",21,29,1,3,0,0,0,0,0,339.483,5,2,2,1,0,0,0,1,0,1,1,3,3.3619,59.22,8,0.428571,"""train"""
15628,"""Bromine""","""BrBr""",-7.2,"""Br2""",0,0,0,0,0,0,0,2,0,159.808,0,0,0,0,0,0,0,0,0,0,0,0,1.6912,0.0,0,0.0,"""test"""


## Featurization

### Node features (40 dims)
| Group | Encoding | Dims |
|---|---|---|
| Atomic number | one-hot over {H,B,C,N,O,F,Si,P,S,Cl,Br,I} + other | 13 |
| Degree | one-hot over {0–5} + other | 7 |
| Total H count | one-hot over {0–4} + other | 6 |
| Formal charge | one-hot over {-2,-1,0,1,2} + other | 6 |
| Aromaticity | binary | 1 |
| Hybridization | one-hot over {SP,SP2,SP3,SP3D,SP3D2} + other | 6 |
| Gasteiger charge | scalar | 1 |

### Edge features (8 dims)
| Feature | Encoding | Dims |
|---|---|---|
| Bond type | one-hot over {single,double,triple,aromatic} + other | 5 |
| Conjugated | binary | 1 |
| In ring | binary | 1 |
| |ΔEN| | Pauling electronegativity difference (scalar) | 1 |

### Molecule-level features (9 dims)
MolWt, LogP, TPSA, H-bond donors, H-bond acceptors, rotatable bonds, ring count, aromatic ring count, frac_csp3

In [3]:
ATOMIC_NUMS = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]  # H B C N O F Si P S Cl Br I

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

HYBRIDIZATIONS = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]

EN_MAP = {
    1: 2.20,   # H
    5: 2.04,   # B
    6: 2.55,   # C
    7: 3.04,   # N
    8: 3.44,   # O
    9: 3.98,   # F
    14: 1.90,  # Si
    15: 2.19,  # P
    16: 2.58,  # S
    17: 3.16,  # Cl
    35: 2.96,  # Br
    53: 2.66,  # I
}


def one_hot(val, choices):
    return [int(val == c) for c in choices] + [int(val not in choices)]


def atom_features(atom):
    charge = atom.GetDoubleProp("_GasteigerCharge")
    if charge != charge:  # replace NaN (rare, unusual valence) with 0
        charge = 0.0
    return (
        one_hot(atom.GetAtomicNum(), ATOMIC_NUMS)          # 13
        + one_hot(atom.GetDegree(), [0, 1, 2, 3, 4, 5])   # 7
        + one_hot(atom.GetTotalNumHs(), [0, 1, 2, 3, 4])  # 6
        + one_hot(atom.GetFormalCharge(), [-2,-1,0,1,2])   # 6
        + [int(atom.GetIsAromatic())]                      # 1
        + one_hot(atom.GetHybridization(), HYBRIDIZATIONS) # 6
        + [charge]                                         # 1
    )  # total: 40


def bond_features(bond, atom_i, atom_j):
    delta_en = abs(
        EN_MAP.get(atom_i.GetAtomicNum(), 0.0)
        - EN_MAP.get(atom_j.GetAtomicNum(), 0.0)
    )
    return (
        one_hot(bond.GetBondType(), BOND_TYPES)                    # 5
        + [int(bond.GetIsConjugated()), int(bond.IsInRing())]      # 2
        + [delta_en]                                               # 1
    )  # total: 8


def mol_features(mol):
    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        rdMolDescriptors.CalcTPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
        rdMolDescriptors.CalcNumRings(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),
    ]


NODE_DIM = 40
EDGE_DIM = 8
MOL_FEAT_DIM = 9


def smiles_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    rdPartialCharges.ComputeGasteigerCharges(mol)
    num_atoms = mol.GetNumAtoms()

    # Real atom features + one zero-initialised virtual node (last row)
    x = torch.cat([
        torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float),
        torch.zeros(1, NODE_DIM),
    ], dim=0)

    src, dst, edge_attrs = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = bond_features(bond, mol.GetAtomWithIdx(i), mol.GetAtomWithIdx(j))
        src += [i, j]
        dst += [j, i]
        edge_attrs += [feat, feat]

    # Virtual node (index num_atoms) connected bidirectionally to every real atom
    vn = num_atoms
    zero_edge = [0.0] * EDGE_DIM
    for i in range(num_atoms):
        src += [i, vn]
        dst += [vn, i]
        edge_attrs += [zero_edge, zero_edge]

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attrs, dtype=torch.float)

    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        mol_feat=torch.tensor([mol_features(mol)], dtype=torch.float),
        y=torch.tensor([y], dtype=torch.float),
    )


print(f"Node feature dim: {NODE_DIM}")
print(f"Edge feature dim: {EDGE_DIM}")
print(f"Mol feature dim:  {MOL_FEAT_DIM}")

Node feature dim: 40
Edge feature dim: 8
Mol feature dim:  9


In [4]:
graphs = []
failed = []
for row in df.iter_rows(named=True):
    g = smiles_to_graph(row["smiles"], row["mpC"])
    if g is None:
        failed.append(row["smiles"])
    else:
        graphs.append(g)

print(f"Graphs built: {len(graphs)}  |  Failed: {len(failed)}")

Graphs built: 3024  |  Failed: 0


In [5]:
torch.save(
    {
        "graphs": graphs,
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "mol_feat_dim": MOL_FEAT_DIM,
    },
    "../data/processed/graphs.pt",
)
print(f"Saved {len(graphs)} graphs to data/processed/graphs.pt")

Saved 3024 graphs to data/processed/graphs.pt
